# Root2Raj Retail Demand Forecasting Benchmark
### Can machine learning beat three simple forecasting rules?

**Root2Raj · Ruturajsinh Zala · Reproducible public-data contribution**

AI-assisted portfolio work. This is a reproducible experiment, not employment, client work, or a claim of independent coding proficiency.

[Original revenue audit](https://www.kaggle.com/code/ruturajsinhzala/root2raj-retail-revenue-and-credit-audit) · [Reusable source and tests](https://github.com/ruturaj1zala123-source/root2raj-retail-operations/tree/main/contributions) · [Public aggregate dataset](https://www.kaggle.com/datasets/ruturajsinhzala/root2raj-retail-audit-evidence)

**Source:** Chen, D. (2015), *Online Retail*, UCI. [Dataset](https://archive.ics.uci.edu/dataset/352/online%2Bretail) · [DOI](https://doi.org/10.24432/C5BW33) · [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Historical UK retailer records, December 2010–December 2011.

## Reproducible input and explicit cleaning
CPU is sufficient. Enable Internet for the original UCI download, or attach the unchanged Online Retail.xlsx. We verify the workbook SHA-256 and 541,909-row schema before analysis. Exact duplicates are removed as an explicit assumption, consistent with the original audit. Identical lines could represent genuine repeats.

A sale has positive quantity, positive price, a valid invoice/date, and a non-C invoice. A credit has negative quantity under a C invoice. Other rows are excluded from these experiments. No customer identifiers appear in outputs.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, urllib.request, zipfile, platform, unittest
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, FileLink
ROOT=Path("/kaggle/working/root2raj-retail-demand-benchmark") if Path("/kaggle/working").is_dir() else Path(os.environ.get("ROOT2RAJ_HOME",str(Path.cwd()/"root2raj-retail-demand-benchmark-run")))
DATA,OUT=ROOT/"data",ROOT/"outputs"
DATA.mkdir(parents=True,exist_ok=True); OUT.mkdir(parents=True,exist_ok=True)
EXPECTED_HASH="43465a06f2ccf7c8b5bd2892bc7defb52f97487934fe93b16ae4c3936424676d"
attached=sorted(Path("/kaggle/input").rglob("Online Retail.xlsx")) if Path("/kaggle/input").is_dir() else []
source=attached[0] if attached else DATA/"Online Retail.xlsx"
if not source.exists():
    archive=DATA/"uci-source.zip"
    urllib.request.urlretrieve("https://archive.ics.uci.edu/static/public/352/online%2Bretail.zip",archive)
    with zipfile.ZipFile(archive) as z:
        names=[n for n in z.namelist() if Path(n).name=="Online Retail.xlsx"]
        if len(names)!=1: raise ValueError("Expected one UCI workbook")
        source.write_bytes(z.read(names[0]))
source_hash=hashlib.sha256(source.read_bytes()).hexdigest()
assert source_hash==EXPECTED_HASH,"Source workbook changed"
raw=pd.read_excel(source,engine="openpyxl")
assert set(raw.columns)=={"InvoiceNo","StockCode","Description","Quantity","InvoiceDate","UnitPrice","CustomerID","Country"}
assert len(raw)==541909
duplicate_count=int(raw.duplicated().sum())
df=raw.drop_duplicates().copy()
df.InvoiceNo=df.InvoiceNo.astype("string").str.strip()
df.StockCode=df.StockCode.astype("string").str.strip()
df.InvoiceDate=pd.to_datetime(df.InvoiceDate,errors="coerce")
valid=(df.InvoiceDate.notna() & df.InvoiceNo.notna() & np.isfinite(df.Quantity) & np.isfinite(df.UnitPrice) & (df.UnitPrice>0) & (df.Quantity!=0))
cancel=df.InvoiceNo.str.upper().str.startswith("C").fillna(False)
df["kind"]="excluded"
df.loc[valid & ~cancel & (df.Quantity>0),"kind"]="sale"
df.loc[valid & cancel & (df.Quantity<0),"kind"]="credit"
assert (df.loc[df.kind.isin(["sale","credit"]),"Quantity"]%1==0).all()
print("Verified source:",source_hash)
display(pd.DataFrame([{"raw_rows":len(raw),"duplicate_rows_removed":duplicate_count,"sale_rows":int((df.kind=="sale").sum()),"credit_rows":int((df.kind=="credit").sum())}]))

## Reusable implementation
The module below is written into the run directory and imported. It is also included in the evidence ZIP, together with meaningful synthetic tests. The complete implementation is visible so readers can inspect or adapt every assumption.

In [ ]:
module_source='"""One-day-ahead benchmark with shifted features and fixed chronological splits."""\nimport numpy as np\nimport pandas as pd\nfrom sklearn.base import clone\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import Ridge\nfrom sklearn.ensemble import HistGradientBoostingRegressor\nfrom threadpoolctl import threadpool_limits\n\nTRAIN_END=pd.Timestamp("2011-06-30")\nVALID_END=pd.Timestamp("2011-08-31")\nTEST_END=pd.Timestamp("2011-11-30")\n\ndef select_products(sales,n=12):\n    # Product selection uses only the initial training period.\n    training=sales[sales.InvoiceDate.dt.normalize()<=TRAIN_END]\n    ranked=training.groupby("StockCode").Quantity.sum().sort_values(ascending=False,kind="stable")\n    return ranked.head(n).index.astype(str).tolist()\n\ndef make_panel(sales,products):\n    days=pd.date_range("2010-12-01",TEST_END,freq="D")\n    data=sales[sales.StockCode.isin(products)].copy()\n    data["date"]=data.InvoiceDate.dt.normalize()\n    sums=data.groupby(["StockCode","date"]).Quantity.sum()\n    index=pd.MultiIndex.from_product([products,days],names=["StockCode","date"])\n    panel=sums.reindex(index,fill_value=0).rename("units").reset_index()\n    return panel.sort_values(["StockCode","date"]).reset_index(drop=True)\n\ndef make_features(panel):\n    panel=panel.sort_values(["StockCode","date"]).reset_index(drop=True).copy()\n    grouped=panel.groupby("StockCode",sort=False).units\n    for lag in (1,7,14,28):\n        panel[f"lag_{lag}"]=grouped.shift(lag)\n    for width in (7,28):\n        panel[f"mean_{width}"]=grouped.transform(lambda s:s.shift(1).rolling(width,min_periods=width).mean())\n    day=panel.date.dt.dayofweek\n    panel["dow_sin"]=np.sin(2*np.pi*day/7)\n    panel["dow_cos"]=np.cos(2*np.pi*day/7)\n    panel["month_sin"]=np.sin(2*np.pi*panel.date.dt.month/12)\n    panel["month_cos"]=np.cos(2*np.pi*panel.date.dt.month/12)\n    dummies=pd.get_dummies(panel.StockCode,prefix="sku",dtype=float)\n    panel=pd.concat([panel,dummies],axis=1)\n    features=["lag_1","lag_7","lag_14","lag_28","mean_7","mean_28","dow_sin","dow_cos","month_sin","month_cos"]+dummies.columns.tolist()\n    return panel.dropna(subset=features).reset_index(drop=True),features\n\ndef metrics(actual,prediction):\n    actual=np.asarray(actual,dtype=float); prediction=np.asarray(prediction,dtype=float)\n    if len(actual)==0 or not np.isfinite(prediction).all() or actual.sum()<=0:\n        raise ValueError("Expected finite predictions and positive aggregate demand")\n    error=np.abs(actual-prediction)\n    return dict(mae_units=float(error.mean()),wape_pct=float(100*error.sum()/actual.sum()))\n\ndef run_benchmark(table,features):\n    train=table[table.date<=TRAIN_END]\n    valid=table[(table.date>TRAIN_END)&(table.date<=VALID_END)]\n    test=table[(table.date>VALID_END)&(table.date<=TEST_END)]\n    assert train.date.max()<valid.date.min() and valid.date.max()<test.date.min()\n    candidates={\n        "Ridge alpha=1":make_pipeline(StandardScaler(),Ridge(alpha=1)),\n        "Ridge alpha=100":make_pipeline(StandardScaler(),Ridge(alpha=100)),\n        "Gradient boosting 15 leaves":HistGradientBoostingRegressor(max_iter=120,max_leaf_nodes=15,learning_rate=.06,loss="absolute_error",early_stopping=False,random_state=42),\n        "Gradient boosting 31 leaves":HistGradientBoostingRegressor(max_iter=120,max_leaf_nodes=31,learning_rate=.06,loss="absolute_error",early_stopping=False,random_state=42)}\n    baselines={"Yesterday":"lag_1","Same weekday last week":"lag_7","Trailing 7-day mean":"mean_7"}\n    validation=[]; valid_predictions={}\n    with threadpool_limits(limits=2):\n        for name,column in baselines.items():\n            pred=valid[column].to_numpy()\n            valid_predictions[name]=pred\n            validation.append(dict(model=name,**metrics(valid.units,pred)))\n        for name,model in candidates.items():\n            model.fit(train[features],train.units)\n            pred=np.clip(model.predict(valid[features]),0,None)\n            valid_predictions[name]=pred\n            validation.append(dict(model=name,**metrics(valid.units,pred)))\n    validation=pd.DataFrame(validation).sort_values(["wape_pct","model"])\n    winner=validation.iloc[0].model\n    # Selection is completed before any test prediction or test score.\n    final_train=table[table.date<=VALID_END]\n    predictions=test[["StockCode","date","units"]].copy()\n    scorecard=[]\n    with threadpool_limits(limits=2):\n        for name in validation.model:\n            if name in baselines:\n                pred=test[baselines[name]].to_numpy()\n            else:\n                fitted=clone(candidates[name]).fit(final_train[features],final_train.units)\n                pred=np.clip(fitted.predict(test[features]),0,None)\n            predictions[name]=pred\n            val=validation.set_index("model").loc[name]\n            scorecard.append(dict(model=name,selected_on_validation=name==winner,\n                validation_mae_units=float(val.mae_units),validation_wape_pct=float(val.wape_pct),\n                test_mae_units=metrics(test.units,pred)["mae_units"],test_wape_pct=metrics(test.units,pred)["wape_pct"]))\n    per_product=[]\n    for sku,g in predictions.groupby("StockCode"):\n        if g.units.sum()>0:\n            for name in validation.model:\n                per_product.append(dict(StockCode=sku,model=name,**metrics(g.units,g[name])))\n    return pd.DataFrame(scorecard),predictions,pd.DataFrame(per_product),winner\n'
(ROOT/'demand_benchmark.py').write_text(module_source,encoding="utf-8")
sys.path.insert(0,str(ROOT))
from demand_benchmark import *
NAVY,TEAL,CORAL,GREY="#162A43","#087F8C","#D56343","#66758A"
plt.rcParams.update({"figure.facecolor":"white","axes.facecolor":"white","axes.spines.top":False,"axes.spines.right":False,"text.color":NAVY,"axes.labelcolor":NAVY,"font.size":11,"axes.titlesize":15,"axes.titleweight":"bold","figure.dpi":130})
pd.set_option("display.float_format",lambda v:f"{v:,.2f}")
def chart(fig,name):
    fig.savefig(OUT/name,dpi=170,bbox_inches="tight")
    plt.show(); plt.close(fig)

## Forecast contract: one day ahead, observed history
**Target:** daily positive units recorded for 12 merchandise-style stock codes (five digits with an optional uppercase suffix). This operational code rule may omit valid products. Credits and non-merchandise-style codes are excluded; the target is not net demand, profitability, or customer demand censored by stockouts.

Products are selected by positive units **only through 30 June 2011**. No future bestsellers enter selection. Calendar days run from 1 December 2010 to 30 November 2011; partial December 2011 is excluded. No-record days become zero observed units, which assumes the extract has no missing trading records.

| Stage | Dates | Use |
|:--|:--|:--|
| Initial training | Through 30 June 2011 | Fit candidate models; select products |
| Validation | 1 July–31 August 2011 | Select model by lowest WAPE |
| Final training | Through 31 August 2011 | Refit with validation history |
| Held-out test | 1 September–30 November 2011 | Report once after selection |

Each forecast uses actual observations through yesterday, including earlier test days as they become available. Model parameters remain fixed during test. This is rolling one-day-ahead evaluation, not a 91-day forecast made on 31 August. First 28 days are warm-up.

In [ ]:
merchandise=df[(df.kind=="sale") & df.StockCode.str.fullmatch(r"\d{5}[A-Z]?",na=False)].copy()
products=select_products(merchandise,n=12)
panel=make_panel(merchandise,products)
table,features=make_features(panel)
display(pd.DataFrame({"selected_stock_code":products}))
display(pd.DataFrame([{"panel_rows":len(panel),"warmup_rows":len(panel)-len(table),"model_rows":len(table),"selected_products":len(products),"recorded_units":float(panel.units.sum())}]))
print("Features:",features)

## Baselines before machine learning
We compare yesterday, the same weekday last week, and the trailing seven-day mean with standardized Ridge regression (α = 1 or 100) and histogram gradient boosting (15 or 31 leaves, 120 iterations, absolute-error loss). All predictions are clipped at zero.

Lag features use 1, 7, 14 and 28 days. Rolling means use **shift(1)** before aggregation. Calendar features are known in advance, and product indicators come from the training-selected catalog. Automatic random early-stopping splits are disabled.

**MAE** averages absolute unit errors across product-days. **WAPE** divides total absolute error by total actual units; it can exceed 100% and gives more weight to high-volume products. We publish per-product errors because aggregate scores can hide weak products.

Methods follow the principles illustrated in [scikit-learn's lagged forecasting example](https://scikit-learn.org/stable/auto_examples/applications/plot_time_series_lagged_features.html). This is an applied benchmark, not a new forecasting algorithm.

In [ ]:
scorecard,predictions,per_product,winner=run_benchmark(table,features)
scorecard.to_csv(OUT/"model_scorecard.csv",index=False)
predictions.to_csv(OUT/"holdout_forecasts.csv",index=False)
per_product.to_csv(OUT/"per_product_errors.csv",index=False)
display(scorecard)
print("Selected using validation only:",winner)
checks={
 "source_hash_verified":source_hash==EXPECTED_HASH,
 "exactly_12_training_selected_products":len(products)==12,
 "validation_after_train":bool(table.loc[table.date<=TRAIN_END,"date"].max()<table.loc[table.date>TRAIN_END,"date"].min()),
 "test_after_validation":bool(predictions.date.min()>VALID_END),
 "test_has_91_days_per_product":bool((predictions.groupby("StockCode").size()==91).all()),
 "finite_nonnegative_predictions":bool(np.isfinite(predictions[scorecard.model]).all().all() and (predictions[scorecard.model]>=0).all().all()),
 "selection_agrees_with_validation_score":winner==scorecard.sort_values(["validation_wape_pct","model"]).iloc[0].model,
 "no_partial_december_in_test":bool(predictions.date.max()==TEST_END)}
assert all(checks.values()),checks

## Measured performance and failure visibility
The highlighted model is the validation winner, even if a different model later scores better on test. We do not replace the selection after seeing test results. Validation and test have different demand patterns, so their scores need not agree.

In [ ]:
fig,ax=plt.subplots(figsize=(12,5.6))
pos=np.arange(len(scorecard)); width=.36
ax.barh(pos-width/2,scorecard.validation_wape_pct,width,label="Validation",color=TEAL)
ax.barh(pos+width/2,scorecard.test_wape_pct,width,label="Held-out test",color=CORAL)
labels=[name+("  ← validation winner" if name==winner else "") for name in scorecard.model]
ax.set_yticks(pos,labels); ax.invert_yaxis()
ax.set(title="Do the models beat simple forecasting rules?",xlabel="WAPE (%) — lower is better")
ax.legend(); chart(fig,"01-model-scorecard.png")
daily=predictions.groupby("date")[["units",winner,"Same weekday last week"]].sum()
fig,ax=plt.subplots(figsize=(12,4.7))
ax.plot(daily.index,daily.units,color=NAVY,label="Observed units",linewidth=1.8)
ax.plot(daily.index,daily[winner],color=TEAL,label="Validation winner",alpha=.8)
ax.plot(daily.index,daily["Same weekday last week"],color=CORAL,label="Weekday baseline",alpha=.6)
ax.set(title="Held-out daily totals across 12 products",ylabel="Recorded positive units",xlabel="")
ax.legend(); chart(fig,"02-daily-forecasts.png")
g=per_product[per_product.model==winner].sort_values("mae_units",ascending=False)
fig,ax=plt.subplots(figsize=(11,4.5))
ax.bar(g.StockCode,g.mae_units,color=TEAL)
ax.set(title="Averages hide product-level errors",ylabel="Test MAE (units per day)",xlabel="Stock code")
chart(fig,"03-product-errors.png")

## Interpretation and next experiment
This extract mixes wholesale and retail activity; large orders, sparse days, promotions and stock availability are not fully observed. The selected catalog and historical period limit generalization. A model can miss a large order even when its ordinary-day forecasts are useful.

Do not translate these errors into achieved inventory savings. A next experiment would need rolling multi-period evaluations, stockout information, lead times, asymmetric shortage costs, and a decision simulation. The fixed historical holdout here is transparent but is not statistical proof that one model will always win.

In [ ]:
selected=scorecard[scorecard.model==winner].iloc[0]
baseline=scorecard[scorecard.model=="Same weekday last week"].iloc[0]
skill=100*(1-selected.test_wape_pct/baseline.test_wape_pct)
display(Markdown(f"**Measured result:** Validation selected **{winner}**. Held-out WAPE is **{selected.test_wape_pct:.2f}%**, compared with **{baseline.test_wape_pct:.2f}%** for the weekday baseline. Relative error reduction is **{skill:.2f}%** (negative means worse)."))
results={"source_sha256":source_hash,"target":"Daily positive units for 12 training-selected merchandise-style codes","validation_winner":winner,"test_wape_pct":float(selected.test_wape_pct),"weekday_baseline_test_wape_pct":float(baseline.test_wape_pct),"relative_error_reduction_vs_weekday_pct":float(skill),"test_period":["2011-09-01","2011-11-30"],"forecast_horizon":"One day ahead with actual observations available through yesterday","selected_stock_codes":products}
(OUT/"results.json").write_text(json.dumps(results,indent=2),encoding="utf-8")
(OUT/"DATA_DICTIONARY.md").write_text("""# Evidence dictionary
model_scorecard.csv: seven candidates; validation and test MAE (units) and WAPE (%). selected_on_validation flags the pre-test choice.
holdout_forecasts.csv: stock code × date × observed units, followed by each model's predicted units. These are daily product aggregates; no customer or invoice IDs.
per_product_errors.csv: test MAE and WAPE by product and model, where product aggregate units are positive.
All forecasts use observations available through the previous day. WAPE can exceed 100%. Derived evidence is CC BY 4.0; code is Apache 2.0.
""",encoding="utf-8")

## Data architecture: contracts, lineage and a queryable gold layer
The unchanged workbook is the bronze snapshot; explicit deduplication and classification form the silver layer. Experiment-specific aggregate tables form gold data products. This is a local reference architecture, not a deployed warehouse or an orchestration service.

Version 1.0.0 contracts declare table grain, types, nullability and nonnegative measures. SQLite STRICT tables enforce those types and primary keys. The notebook checks database integrity, row counts, and duplicate-key rejection. A lineage manifest records the workbook hash, implementation hash, CSV hashes and transformation edges. Customer and invoice IDs are excluded from the gold schema. Raw and silver layers stay in memory or source storage; only gold is exported.

In [ ]:
architecture_source='"""Export versioned gold data products with lineage and executable SQL contracts."""\nfrom pathlib import Path\nfrom contextlib import closing\nimport hashlib,json,sqlite3\nimport pandas as pd\nimport numpy as np\n\nCONTRACT_VERSION="1.0.0"\n\ndef export_data_product(out,tables,primary_keys,source_hash,implementation,experiment):\n    out=Path(out)\n    schema={}; ddl=[]; checks={}\n    assert all("CustomerID" not in t.columns and "InvoiceNo" not in t.columns for t in tables.values())\n    for name,table in tables.items():\n        if not name.replace("_","").isalnum(): raise ValueError("Invalid table identifier")\n        keys=primary_keys[name]\n        if not set(keys).issubset(table.columns): raise ValueError("Missing primary key")\n        if table.empty or table.isna().any().any() or table.duplicated(keys).any():\n            raise ValueError(f"{name}: empty table, null values or duplicate primary key")\n        fields=[]; clauses=[]\n        for col in table.columns:\n            quoted=\'"\'+str(col).replace(\'"\',\'""\')+\'"\'\n            dtype=table[col].dtype\n            sql_type="INTEGER" if pd.api.types.is_integer_dtype(dtype) or pd.api.types.is_bool_dtype(dtype) else "REAL" if pd.api.types.is_numeric_dtype(dtype) else "TEXT"\n            if sql_type in ("INTEGER","REAL") and not np.isfinite(table[col].astype(float)).all():\n                raise ValueError(f"{name}.{col}: non-finite numeric value")\n            nonnegative=sql_type!="TEXT" and (col.endswith("_gbp") or "units" in col or "wape" in col or "coverage" in col or col=="window_days")\n            if nonnegative and (table[col]<0).any(): raise ValueError(f"{name}.{col}: negative value")\n            clause=f"{quoted} {sql_type} NOT NULL"\n            if nonnegative: clause+=f" CHECK ({quoted} >= 0)"\n            clauses.append(clause)\n            fields.append(dict(name=str(col),sql_type=sql_type,nullable=False,nonnegative=nonnegative))\n        key_sql=", ".join(\'"\'+str(k).replace(\'"\',\'""\')+\'"\' for k in keys)\n        statement=f\'CREATE TABLE "{name}" (\\n  \'+",\\n  ".join(clauses)+f",\\n  PRIMARY KEY ({key_sql})\\n) STRICT;"\n        ddl.append(statement)\n        schema[name]={"grain":keys,"row_count":len(table),"fields":fields}\n    with closing(sqlite3.connect(out/"gold.sqlite")) as con:\n        for name,table in tables.items():\n            con.execute(f\'DROP TABLE IF EXISTS "{name}"\')\n            con.execute(ddl[list(tables).index(name)])\n            normalized=table.copy()\n            for col in normalized.columns:\n                if pd.api.types.is_datetime64_any_dtype(normalized[col].dtype):\n                    normalized[col]=normalized[col].dt.strftime("%Y-%m-%d")\n            normalized.to_sql(name,con,index=False,if_exists="append")\n            count=con.execute(f\'SELECT COUNT(*) FROM "{name}"\').fetchone()[0]\n            assert count==len(table)\n            checks[name+"_sql_row_count"]=True\n            # The primary-key rejection is tested against a real exported row.\n            quoted_columns=", ".join(\'"\'+str(c).replace(\'"\',\'""\')+\'"\' for c in table.columns)\n            con.execute("SAVEPOINT contract_probe")\n            try:\n                con.execute(f\'INSERT INTO "{name}" ({quoted_columns}) SELECT {quoted_columns} FROM "{name}" LIMIT 1\')\n            except sqlite3.IntegrityError:\n                checks[name+"_duplicate_key_rejected"]=True\n            else:\n                raise AssertionError("SQL primary-key contract failed")\n            finally:\n                con.execute("ROLLBACK TO contract_probe")\n                con.execute("RELEASE contract_probe")\n        checks["sql_integrity_check"]=con.execute("PRAGMA integrity_check").fetchone()[0]=="ok"\n        con.commit()\n    (out/"gold_schema.sql").write_text("\\n\\n".join(ddl),encoding="utf-8")\n    contracts={"schema_version":CONTRACT_VERSION,"layer":"gold","tables":schema,\n        "freshness":"Static historical snapshot; no scheduled ingestion or live freshness claim",\n        "breaking_change_rule":"Increase major version if a field type, primary key, target, or denominator changes"}\n    (out/"data_contracts.json").write_text(json.dumps(contracts,indent=2),encoding="utf-8")\n    lineage={"manifest_version":CONTRACT_VERSION,"experiment":experiment,\n        "source":{"name":"UCI Online Retail","doi":"10.24432/C5BW33","license":"CC BY 4.0","workbook_sha256":source_hash},\n        "implementation_sha256":hashlib.sha256(implementation.encode()).hexdigest(),\n        "layers":[\n            {"id":"bronze","grain":"Original invoice line","operation":"Unchanged workbook; source hash and schema verification"},\n            {"id":"silver","grain":"Deduplicated invoice line","operation":"Exact duplicate removal and explicit sale/credit classification"},\n            {"id":"experiment","grain":"Defined in notebook","operation":experiment},\n            {"id":"gold","grain":"Declared primary keys","operation":"Aggregate evidence exported under executable SQLite STRICT contracts"}],\n        "edges":[["bronze","silver"],["silver","experiment"],["experiment","gold"]],\n        "privacy":"No invoice or customer identifiers in gold outputs; source workbook is not redistributed",\n        "tables":[{"name":name,"csv_sha256":hashlib.sha256((out/(name+".csv")).read_bytes()).hexdigest(),"rows":len(t)} for name,t in tables.items()]}\n    (out/"lineage.json").write_text(json.dumps(lineage,indent=2),encoding="utf-8")\n    return checks\n'
(ROOT/"evidence_contracts.py").write_text(architecture_source,encoding="utf-8")
from evidence_contracts import export_data_product
architecture_checks=export_data_product(OUT,{"model_scorecard":scorecard,"holdout_forecasts":predictions,"per_product_errors":per_product},{"model_scorecard":["model"],"holdout_forecasts":["StockCode","date"],"per_product_errors":["StockCode","model"]},source_hash,module_source+architecture_source,'One-day-ahead demand benchmark with training-only catalog selection and chronological holdout')
checks.update(architecture_checks)
display(pd.DataFrame([architecture_checks]))
print("Exported gold.sqlite, gold_schema.sql, data_contracts.json and lineage.json")

## Synthetic tests and artifact export
These tests exercise behavior that can otherwise produce plausible but wrong results. Passing them supports implementation correctness under the stated definitions; it cannot establish that those definitions match the retailer's real processes.

In [ ]:
test_source='import unittest\nimport numpy as np\nimport pandas as pd\nfrom demand_benchmark import make_features,select_products,metrics\n\nclass DemandTests(unittest.TestCase):\n    def panel(self):\n        return pd.DataFrame({"StockCode":["A"]*70,"date":pd.date_range("2011-01-01",periods=70),"units":np.arange(70,dtype=float)})\n    def test_shifted_lags_and_mean(self):\n        f,_=make_features(self.panel())\n        first=f.iloc[0]\n        self.assertEqual(first.lag_1,27)\n        self.assertEqual(first.lag_7,21)\n        self.assertEqual(first.mean_7,np.mean(np.arange(21,28)))\n        self.assertEqual(first.mean_28,np.mean(np.arange(28)))\n    def test_future_changes_do_not_change_past_features(self):\n        a=self.panel(); b=a.copy()\n        b.loc[b.date>=pd.Timestamp("2011-02-20"),"units"]=9999\n        fa,columns=make_features(a); fb,_=make_features(b)\n        mask=fa.date<=pd.Timestamp("2011-02-20")\n        np.testing.assert_allclose(fa.loc[mask,columns],fb.loc[mask,columns])\n    def test_product_selection_ignores_future(self):\n        sales=pd.DataFrame({"StockCode":["A","B","B"],"InvoiceDate":pd.to_datetime(["2011-01-01","2011-01-01","2011-10-01"]),"Quantity":[10,2,100000]})\n        self.assertEqual(select_products(sales,1),["A"])\n    def test_product_lags_do_not_cross_boundaries(self):\n        a=self.panel(); b=a.copy(); b.StockCode="B"; b.units=1000\n        f,_=make_features(pd.concat([a,b]))\n        self.assertTrue((f.loc[f.StockCode=="B","lag_1"]==1000).all())\n    def test_wape_and_mae(self):\n        self.assertEqual(metrics([0,10],[2,8]),dict(mae_units=2.,wape_pct=40.))\n        with self.assertRaises(ValueError): metrics([0],[0])\n'
(ROOT/'test_demand_benchmark.py').write_text(test_source,encoding="utf-8")
exec(compile(test_source,'test_demand_benchmark.py',"exec"))
architecture_tests='import unittest,tempfile,os,json,sqlite3\nfrom pathlib import Path\nfrom contextlib import closing\nimport pandas as pd\nfrom evidence_contracts import export_data_product\n\nclass ContractTests(unittest.TestCase):\n    def setUp(self):\n        self.temp=tempfile.TemporaryDirectory(dir=os.environ.get("ROOT2RAJ_TEST_TEMP",os.environ.get("TEMP",".")))\n        self.out=Path(self.temp.name)\n    def tearDown(self):\n        self.temp.cleanup()\n    def export(self,table):\n        table.to_csv(self.out/"facts.csv",index=False)\n        return export_data_product(self.out,{"facts":table},{"facts":["key"]},"a"*64,"verified implementation","Synthetic contract test")\n    def test_valid_export_and_database_key_enforcement(self):\n        checks=self.export(pd.DataFrame({"key":["A","B"],"units":[2,3]}))\n        self.assertTrue(all(checks.values()))\n        with closing(sqlite3.connect(self.out/"gold.sqlite")) as con:\n            self.assertEqual(con.execute("SELECT SUM(units) FROM facts").fetchone()[0],5)\n            with self.assertRaises(sqlite3.IntegrityError):\n                con.execute("INSERT INTO facts VALUES (\'C\',-1)")\n            with self.assertRaises(sqlite3.IntegrityError):\n                con.execute("INSERT INTO facts VALUES (\'C\',NULL)")\n            with self.assertRaises(sqlite3.IntegrityError):\n                con.execute("INSERT INTO facts VALUES (\'C\',\'wrong type\')")\n        contracts=json.loads((self.out/"data_contracts.json").read_text())\n        self.assertEqual(contracts["schema_version"],"1.0.0")\n        lineage=json.loads((self.out/"lineage.json").read_text())\n        self.assertEqual(len(lineage["tables"][0]["csv_sha256"]),64)\n    def test_duplicate_primary_key_rejected(self):\n        with self.assertRaises(ValueError): self.export(pd.DataFrame({"key":["A","A"],"units":[1,2]}))\n    def test_null_measure_rejected(self):\n        with self.assertRaises(ValueError): self.export(pd.DataFrame({"key":["A"],"units":[None]}))\n    def test_negative_measure_rejected(self):\n        with self.assertRaises(ValueError): self.export(pd.DataFrame({"key":["A"],"units":[-1]}))\n    def test_nonfinite_measure_rejected(self):\n        with self.assertRaises(ValueError): self.export(pd.DataFrame({"key":["A"],"units":[float("inf")]}))\n'
(ROOT/"test_evidence_contracts.py").write_text(architecture_tests,encoding="utf-8")
os.environ["ROOT2RAJ_TEST_TEMP"]=str(ROOT)
exec(compile(architecture_tests,"test_evidence_contracts.py","exec"))
suite=unittest.TestSuite([unittest.defaultTestLoader.loadTestsFromTestCase(DemandTests),unittest.defaultTestLoader.loadTestsFromTestCase(ContractTests)])
test_result=unittest.TextTestRunner(verbosity=2).run(suite)
assert test_result.wasSuccessful()
checks["synthetic_tests_passed"]=test_result.wasSuccessful()
checks["synthetic_test_count"]=test_result.testsRun
assert all(v for k,v in checks.items() if k!="synthetic_test_count"),checks
(OUT/"validation.json").write_text(json.dumps(checks,indent=2),encoding="utf-8")
environment={"python":platform.python_version(),"pandas":pd.__version__,"numpy":np.__version__,"matplotlib":matplotlib.__version__}
try:
    import sklearn
    environment["scikit-learn"]=sklearn.__version__
except ImportError: pass
(OUT/"environment.json").write_text(json.dumps(environment,indent=2),encoding="utf-8")
(OUT/"README.md").write_text("Root2Raj AI-assisted public-data experiment. Source: Chen, D. (2015), UCI Online Retail, DOI 10.24432/C5BW33, CC BY 4.0. Source workbook SHA-256: "+source_hash+"\nRead the notebook for assumptions, limitations and interpretation. Derived evidence is CC BY 4.0; code is Apache 2.0.\n",encoding="utf-8")
(ROOT/"requirements.txt").write_text("pandas>=2.2,<4\nnumpy>=1.26,<3\nopenpyxl>=3.1,<4\nmatplotlib>=3.8,<4\nscikit-learn>=1.4,<2\nthreadpoolctl>=3.1,<4\n",encoding="utf-8")
package=ROOT/"root2raj-retail-demand-benchmark-evidence.zip"
with zipfile.ZipFile(package,"w",zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUT.iterdir()):
        if path.is_file(): z.write(path,"evidence/"+path.name)
    for name in ['demand_benchmark.py','test_demand_benchmark.py',"requirements.txt","evidence_contracts.py","test_evidence_contracts.py"]:
        z.write(ROOT/name,"source/"+name)
display(pd.DataFrame([checks]))
display(FileLink(str(package)))
print("Evidence includes aggregate tables, results, validation, environment, charts, source and tests.")